In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchmetrics.regression import R2Score

from rvsablation.utils import PrecomputedLatentDataset, seed_everything

from pathlib import Path

import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_TYPE = "ae"
RUN_NAME = "fsq_videovae_run_1"
EPOCHS = 50
LR = 1e-3
BATCH = 2048
NUM_TUBELETS = 4

rng = seed_everything(42)

🌱 Global seed set to 42


# Linear Physical State Probe

We train a linear probe over the flattened learned represenations to predict the $24$-dim. physical state/observation vector from DMC Walker.

In [2]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for batch in dataloader:
        latents = batch['latents'].to(device)
        latents = latents.reshape(latents.shape[0], -1)
        targets = batch['states'].to(device)

        optimizer.zero_grad()
        preds = model(latents)
        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate_epoch(model, dataloader, criterion, r2_metric, device):
    model.eval()
    total_loss = 0.0
    r2_metric.reset()
    
    with torch.no_grad():
        for batch in dataloader:
            latents = batch['latents'].to(device)
            latents = latents.reshape(latents.shape[0], -1)
            targets = batch['states'].to(device)
            
            preds = model(latents)
            loss = criterion(preds, targets)
            total_loss += loss.item()
            
            r2_metric.update(preds, targets)
            
    avg_loss = total_loss / len(dataloader)
    
    r2_per_dim = r2_metric.compute() 
    mean_r2 = r2_per_dim.mean().item()
    
    return avg_loss, mean_r2, r2_per_dim.cpu().numpy()

In [3]:
print(f"⚙️ Using latents from {RUN_NAME}")

main_dir = Path("..") / f"{MODEL_TYPE}_runs" / RUN_NAME

data_dir = main_dir / "latents"
data_dir = data_dir.resolve()

print(f"📁 Loading latents from {data_dir}")

train_dataset = PrecomputedLatentDataset(data_dir / "training")
test_dataset = PrecomputedLatentDataset(data_dir / "testing")

train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH, shuffle=False)

⚙️ Using latents from fsq_videovae_run_1
📁 Loading latents from /home/ron/dev/croce_lab/reprod_vs_semantic/ae_runs/fsq_videovae_run_1/latents


In [5]:
TASK_DIM = train_dataset[0]["states"].shape[0]
REP_N, REP_D = train_dataset[0]["latents"].shape
IN_DIM = REP_N * REP_D
print(f"Predicting state observations of {TASK_DIM} dimensions via linear probe over inputs of {IN_DIM} dimensions.")

Predicting state observations of 24 dimensions via linear probe over inputs of 1176 dimensions.


In [6]:
probe = nn.Linear(IN_DIM, TASK_DIM).to(DEVICE)
optimizer = optim.Adam(probe.parameters(), lr=LR)
criterion = nn.MSELoss()
metric = R2Score(multioutput="raw_values").to(DEVICE)

print(f"--- Training {RUN_NAME.upper()} State Decoding Probe ---")
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(probe, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_r2, r2_breakdown = evaluate_epoch(probe, test_loader, criterion, metric, DEVICE)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f} | Val Mean R2: {val_r2:.4f}")

print("\n--- Final Dimensional R2 Breakdown ---")
# This matches up with the 24 dimensions of the Walker state
for i, r2_val in enumerate(r2_breakdown):
    print(f"Dim {i:02d}: {r2_val:.4f}")
print(f"Mean R^2: {val_r2}")

os.makedirs(main_dir / "state_probe", exist_ok=True)
torch.save(probe.state_dict(), main_dir / "state_probe" / "linear_probe.pt")
print(f"\n\n💾 Saved probe to {main_dir / "state_probe" / "linear_probe.pt"}.")

--- Training FSQ_VIDEOVAE_RUN_1 State Decoding Probe ---
Epoch 001 | Train MSE: 19.2178 | Val MSE: 16.8998 | Val Mean R2: -0.0117
Epoch 005 | Train MSE: 11.5942 | Val MSE: 12.1487 | Val Mean R2: 0.6492
Epoch 010 | Train MSE: 10.1978 | Val MSE: 10.9200 | Val Mean R2: 0.7333
Epoch 015 | Train MSE: 9.5806 | Val MSE: 10.4262 | Val Mean R2: 0.7502
Epoch 020 | Train MSE: 9.1703 | Val MSE: 10.1665 | Val Mean R2: 0.7565
Epoch 025 | Train MSE: 9.1419 | Val MSE: 10.0166 | Val Mean R2: 0.7578
Epoch 030 | Train MSE: 8.9138 | Val MSE: 9.9337 | Val Mean R2: 0.7598
Epoch 035 | Train MSE: 8.7908 | Val MSE: 9.8891 | Val Mean R2: 0.7590
Epoch 040 | Train MSE: 8.8319 | Val MSE: 9.8595 | Val Mean R2: 0.7614
Epoch 045 | Train MSE: 8.6653 | Val MSE: 9.8302 | Val Mean R2: 0.7588
Epoch 050 | Train MSE: 8.7031 | Val MSE: 9.8180 | Val Mean R2: 0.7608

--- Final Dimensional R2 Breakdown ---
Dim 00: 0.8652
Dim 01: 0.7238
Dim 02: 0.6974
Dim 03: 0.6767
Dim 04: 0.4979
Dim 05: 0.3233
Dim 06: 0.6555
Dim 07: 0.4598
Dim